In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
df_encoded = pd.read_csv("data/processed_accidents_2019.csv")
results = []

support_values = [0.01, 0.02, 0.05, 0.1]
maxlen_values = [2, 3, 4]

for s in support_values:
    for ml in maxlen_values:
        # Run Apriori
        freq = apriori(df_encoded, 
                       min_support=s, 
                       use_colnames=True, 
                       max_len=ml)
        
        # If no itemsets, skip
        if freq.empty:
            results.append({
                "min_support": s,
                "max_len": ml,
                "num_itemsets": 0,
                "num_rules": 0,
                "avg_confidence": 0,
                "avg_lift": 0
            })
            continue
        
        # Generate rules
        rules = association_rules(freq, metric="confidence", min_threshold=0.4)

        # Collect rule statistics
        if rules.empty:
            avg_conf = 0
            avg_lift = 0
        else:
            avg_conf = rules["confidence"].mean()
            avg_lift = rules["lift"].mean()

        results.append({
            "min_support": s,
            "max_len": ml,
            "num_itemsets": len(freq),
            "num_rules": len(rules),
            "avg_confidence": round(avg_conf, 3),
            "avg_lift": round(avg_lift, 3)
        })
        print(f"rules for parameters (support_value = {s}, max_length = {ml}) : \n", rules)
        print("-"*80)

# Convert to DataFrame for comparison
comparison_df = pd.DataFrame(results)
print(comparison_df)


rules for parameters (support_value = 0.01, max_length = 2) : 
            antecedents                   consequents  antecedent support  \
0           (Daylight)                   (Afternoon)            0.710514   
1          (Afternoon)                    (Daylight)            0.333302   
2          (Afternoon)                         (Dry)            0.333302   
3          (Afternoon)                        (Fine)            0.333302   
4          (Afternoon)  (RoadType_SingleCarriageway)            0.333302   
..                 ...                           ...                 ...   
223            (Urban)                  (Speed_0_30)            0.676669   
224              (Wet)                  (Speed_0_30)            0.274461   
225      (Speed_30_50)                       (Urban)            0.125383   
226  (Surface_Unknown)                       (Urban)            0.011477   
227              (Wet)                       (Urban)            0.274461   

     consequent support